# Chapter 6: Regression Analysis

## Learning Objectives
- Master linear and polynomial regression
- Understand regression evaluation metrics
- Handle overfitting and underfitting
- Build practical regression models

## Table of Contents
1. [Linear Regression Fundamentals](#linear-regression)
2. [Multiple Linear Regression](#multiple-regression)
3. [Polynomial Regression](#polynomial)
4. [Evaluation Metrics](#metrics)
5. [Real-world Projects](#projects)

In [ ]:
# Essential imports for regression
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import seaborn as sns

print("📈 Chapter 6: Regression Analysis")
print("✅ All libraries imported successfully!")

In [ ]:
# Simple Linear Regression
print("--- Simple Linear Regression ---")

# Load house data
house_data = pd.read_csv('../datasets/sample_house_data.csv')

# Simple regression: predict price from size only
X_simple = house_data[['Size']]
y = house_data['Price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_simple, y, test_size=0.3, random_state=42)

# Train model
simple_model = LinearRegression()
simple_model.fit(X_train, y_train)

# Make predictions
y_pred = simple_model.predict(X_test)

# Model parameters
print(f"Intercept (base price): ${simple_model.intercept_:.2f}")
print(f"Slope (price per sq ft): ${simple_model.coef_[0]:.2f} per sq ft")

# Visualize the regression line
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, alpha=0.7, label='Actual')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Regression Line')
plt.xlabel('House Size (sq ft)')
plt.ylabel('Price ($)')
plt.title('Simple Linear Regression: House Size vs Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nMean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.3f} ({r2*100:.1f}% of variance explained)")

In [ ]:
# Multiple Linear Regression
print("--- Multiple Linear Regression ---")

# Use multiple features
X_multi = house_data[['Size', 'Bedrooms', 'Bathrooms', 'Age']]
y = house_data['Price']

# Split data
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y, test_size=0.3, random_state=42
)

# Train model
multi_model = LinearRegression()
multi_model.fit(X_train_multi, y_train_multi)

# Make predictions
y_pred_multi = multi_model.predict(X_test_multi)

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': multi_model.coef_,
    'Abs_Coefficient': np.abs(multi_model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("Feature Importance (by coefficient magnitude):")
print(feature_importance)

# Compare models
mse_simple = mean_squared_error(y_test, simple_model.predict(X_test))
mse_multi = mean_squared_error(y_test_multi, y_pred_multi)
r2_simple = r2_score(y_test, simple_model.predict(X_test))
r2_multi = r2_score(y_test_multi, y_pred_multi)

print(f"\nModel Comparison:")
print(f"Simple Regression - MSE: {mse_simple:.2f}, R²: {r2_simple:.3f}")
print(f"Multiple Regression - MSE: {mse_multi:.2f}, R²: {r2_multi:.3f}")
print(f"Improvement in R²: {r2_multi - r2_simple:.3f}")

# Visualize actual vs predicted
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, simple_model.predict(X_test), alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Simple Regression (R² = {r2_simple:.3f})')

plt.subplot(1, 2, 2)
plt.scatter(y_test_multi, y_pred_multi, alpha=0.7)
plt.plot([y_test_multi.min(), y_test_multi.max()], [y_test_multi.min(), y_test_multi.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Multiple Regression (R² = {r2_multi:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Polynomial Regression
print("--- Polynomial Regression ---")

# Generate some non-linear data for demonstration
np.random.seed(42)
X_poly_demo = np.linspace(0, 10, 50).reshape(-1, 1)
y_poly_demo = 0.5 * X_poly_demo.ravel()**2 + 2 * X_poly_demo.ravel() + 1 + np.random.normal(0, 5, 50)

# Split data
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly_demo, y_poly_demo, test_size=0.3, random_state=42
)

# Compare different polynomial degrees
degrees = [1, 2, 3, 5]
colors = ['blue', 'red', 'green', 'orange']

plt.figure(figsize=(15, 4))

for i, degree in enumerate(degrees):
    plt.subplot(1, 4, i+1)
    
    # Create polynomial features
    poly_features = PolynomialFeatures(degree=degree)
    X_train_poly_transformed = poly_features.fit_transform(X_train_poly)
    X_test_poly_transformed = poly_features.transform(X_test_poly)
    
    # Train model
    poly_model = LinearRegression()
    poly_model.fit(X_train_poly_transformed, y_train_poly)
    
    # Predictions
    y_pred_poly = poly_model.predict(X_test_poly_transformed)
    r2_poly = r2_score(y_test_poly, y_pred_poly)
    
    # Plot
    plt.scatter(X_test_poly, y_test_poly, alpha=0.7, color='black', s=20)
    
    # Create smooth curve for visualization
    X_smooth = np.linspace(X_poly_demo.min(), X_poly_demo.max(), 100).reshape(-1, 1)
    X_smooth_transformed = poly_features.transform(X_smooth)
    y_smooth = poly_model.predict(X_smooth_transformed)
    
    plt.plot(X_smooth, y_smooth, color=colors[i], linewidth=2)
    plt.title(f'Degree {degree}\nR² = {r2_poly:.3f}')
    plt.xlabel('X')
    if i == 0:
        plt.ylabel('Y')

plt.tight_layout()
plt.show()

print("\n📊 Notice how higher degrees can fit the training data better,")
print("but may overfit (especially degree 5 - too wiggly!)")

In [ ]:
# Comprehensive Evaluation Metrics
print("--- Regression Evaluation Metrics ---")

def evaluate_regression(y_true, y_pred, model_name):
    """Calculate and display regression metrics"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{model_name} Performance:")
    print(f"  Mean Squared Error (MSE): {mse:.2f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"  Mean Absolute Error (MAE): {mae:.2f}")
    print(f"  R² Score: {r2:.3f} ({r2*100:.1f}% variance explained)")
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# Evaluate our models
simple_metrics = evaluate_regression(
    y_test, simple_model.predict(X_test), "Simple Linear Regression"
)

multi_metrics = evaluate_regression(
    y_test_multi, multi_model.predict(X_test_multi), "Multiple Linear Regression"
)

# Create comparison chart
metrics_df = pd.DataFrame({
    'Simple': simple_metrics,
    'Multiple': multi_metrics
})

print("\nMetrics Comparison:")
print(metrics_df)

# Visualize residuals (errors)
plt.figure(figsize=(12, 5))

# Residuals for simple model
plt.subplot(1, 2, 1)
residuals_simple = y_test - simple_model.predict(X_test)
plt.scatter(simple_model.predict(X_test), residuals_simple, alpha=0.7)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Simple Model Residuals')

# Residuals for multiple model
plt.subplot(1, 2, 2)
residuals_multi = y_test_multi - multi_model.predict(X_test_multi)
plt.scatter(multi_model.predict(X_test_multi), residuals_multi, alpha=0.7)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Multiple Model Residuals')

plt.tight_layout()
plt.show()

print("\n🎯 Good models have residuals randomly scattered around zero")
print("🔍 Patterns in residuals suggest the model is missing something")

## Key Concepts Summary

### Types of Regression:
- **Simple Linear**: One feature, straight line relationship
- **Multiple Linear**: Multiple features, linear relationships
- **Polynomial**: Curved relationships, higher-order terms

### Important Metrics:
- **MSE**: Mean Squared Error (lower is better)
- **RMSE**: Root Mean Squared Error (same units as target)
- **MAE**: Mean Absolute Error (robust to outliers)
- **R²**: Coefficient of Determination (0-1, higher is better)

### Model Selection Tips:
1. Start simple, add complexity gradually
2. More features ≠ always better (overfitting risk)
3. Check residual plots for patterns
4. Consider the bias-variance tradeoff

## Practice Exercises

### Exercise 1: Feature Engineering
- Create new features (e.g., rooms per bathroom, age squared)
- Test if they improve model performance
- Which engineered features are most useful?

### Exercise 2: Overfitting Detection
- Try polynomial regression with very high degrees (8, 10, 15)
- Compare training vs testing performance
- At what point does overfitting occur?

### Exercise 3: Real Estate Predictor
- Build the best possible house price predictor
- Try different combinations of features
- Document your model selection process

---

## 🎉 Chapter Complete!
**Next**: Move on to Chapter 7 - Classification